# Homework #5: model deployment
###Question 1-4 


In [0]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS `gr5069`.`sl5919`.`model1_predictions` (
        raceId             INT,
        driverId           INT,
        actual_position    INT,
        predicted_position DOUBLE,
        model_name         STRING,
        created_at         TIMESTAMP
    )
""")

spark.sql("""
    CREATE TABLE IF NOT EXISTS `gr5069`.`sl5919`.`model2_predictions` (
        raceId             INT,
        driverId           INT,
        actual_position    INT,
        predicted_position DOUBLE,
        model_name         STRING,
        created_at         TIMESTAMP
    )
""")

print("✅ Tables created:")
print("  - gr5069.sl5919.model1_predictions")
print("  - gr5069.sl5919.model2_predictions")

In [0]:
#Feature Engineerin
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, explained_variance_score
import mlflow
import mlflow.sklearn
import tempfile, os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

df_pit_feats = (
    df_pit_stops
    .withColumn("duration_sec",
        F.when(F.col("duration").rlike(r"^\d+(\.\d+)?$"),
               F.col("duration").cast("double"))
         .otherwise(F.lit(None).cast("double")))
    .groupBy("raceId", "driverId")
    .agg(
        F.round(F.avg("duration_sec"), 3).alias("avg_pit_sec"),
        F.count("*").cast("int").alias("n_pit_stops")
    )
)

df_races_clean = df_races.select(
    "raceId",
    F.col("year").cast("int").alias("year"),
    F.col("round").cast("int").alias("round"),
    F.to_date(F.col("date"), "yyyy-MM-dd").alias("race_date")
)

df_drivers_clean = df_drivers.select(
    "driverId",
    F.to_date(F.col("dob"), "yyyy-MM-dd").alias("dob")
)

df_standings_clean = df_standings.select(
    "raceId", "driverId",
    F.col("points").cast("double").alias("standing_points"),
    F.col("position").cast("int").alias("standing_position"),
    F.col("wins").cast("int").alias("standing_wins")
)

df_results_clean = df_results.select(
    "raceId", "driverId",
    F.col("positionOrder").cast("int").alias("finishing_position"),
    F.col("grid").cast("int").alias("grid_position"),
    F.col("laps").cast("int").alias("laps_completed")
)

df_model = (
    df_results_clean
    .join(df_races_clean,     on="raceId",              how="left")
    .join(df_drivers_clean,   on="driverId",             how="left")
    .join(df_pit_feats,       on=["raceId", "driverId"], how="left")
    .join(df_standings_clean, on=["raceId", "driverId"], how="left")
    .withColumn("driver_age",
        F.floor(F.datediff(F.col("race_date"), F.col("dob")) / 365.25))
    .select(
        "raceId", "driverId",
        "grid_position", "driver_age", "laps_completed",
        "avg_pit_sec", "n_pit_stops",
        "standing_points", "standing_position", "standing_wins",
        "round", "year",
        "finishing_position"
    )
    .filter(F.col("finishing_position").isNotNull())
)

df = df_model.toPandas()
df = df.fillna(df.median(numeric_only=True))

feature_cols = [
    "grid_position", "driver_age", "laps_completed",
    "avg_pit_sec", "n_pit_stops",
    "standing_points", "standing_position", "standing_wins",
    "round", "year"
]

X_train, X_test, y_train, y_test = train_test_split(
    df[feature_cols], df["finishing_position"],
    test_size=0.2, random_state=42
)

print(f"✅ Feature engineering done: {df.shape}")
print(f"   Train: {X_train.shape} | Test: {X_test.shape}")


In [0]:
# Model 1 — Random Forest

mlflow.set_experiment("/Users/sl5919@columbia.edu/hw5_f1_model1_random_forest")

with mlflow.start_run(run_name="random_forest_v1") as run1:

    params1 = {"n_estimators": 300, "max_depth": 20, "min_samples_leaf": 2}

    rf = RandomForestRegressor(**params1, random_state=42)
    rf.fit(X_train, y_train)
    preds1 = rf.predict(X_test)


    [mlflow.log_param(k, v) for k, v in params1.items()]
    mlflow.log_param("feature_cols", str(feature_cols))

    #metrics
    mse1  = mean_squared_error(y_test, preds1)
    rmse1 = np.sqrt(mse1)
    mae1  = mean_absolute_error(y_test, preds1)
    r2_1  = r2_score(y_test, preds1)
    var1  = explained_variance_score(y_test, preds1)

    mlflow.log_metric("rmse", rmse1)
    mlflow.log_metric("mae",  mae1)
    mlflow.log_metric("r2",   r2_1)
    mlflow.log_metric("explained_variance", var1)

    mlflow.sklearn.log_model(rf, "random_forest_model")

    # Artifact 1
    importance1 = pd.DataFrame({
        "Feature": feature_cols,
        "Importance": rf.feature_importances_
    }).sort_values("Importance", ascending=False)
    tmp1 = tempfile.NamedTemporaryFile(suffix=".csv", delete=False)
    importance1.to_csv(tmp1.name, index=False)
    mlflow.log_artifact(tmp1.name, "feature_importance.csv")
    os.unlink(tmp1.name)

    # Artifact 2: Residual plot
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(preds1, y_test - preds1, alpha=0.3)
    ax.axhline(0, color="red", linestyle="--")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Residual")
    ax.set_title("Residual Plot — Random Forest")
    tmp2 = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
    fig.savefig(tmp2.name); plt.close(fig)
    mlflow.log_artifact(tmp2.name, "residual_plot.png")
    os.unlink(tmp2.name)

    print(f"✅ Model 1 - Random Forest")
    print(f"   RMSE: {rmse1:.4f} | MAE: {mae1:.4f} | R²: {r2_1:.4f} | Var: {var1:.4f}")

#Gradient Boosting

mlflow.set_experiment("/Users/sl5919@columbia.edu/hw5_f1_model2_gbt")

with mlflow.start_run(run_name="gbt_v1") as run2:

    params2 = {"n_estimators": 200, "max_depth": 5, "learning_rate": 0.1}

    gbt = GradientBoostingRegressor(**params2, random_state=42)
    gbt.fit(X_train, y_train)
    preds2 = gbt.predict(X_test)

    [mlflow.log_param(k, v) for k, v in params2.items()]
    mlflow.log_param("feature_cols", str(feature_cols))

    # metrics
    mse2  = mean_squared_error(y_test, preds2)
    rmse2 = np.sqrt(mse2)
    mae2  = mean_absolute_error(y_test, preds2)
    r2_2  = r2_score(y_test, preds2)
    var2  = explained_variance_score(y_test, preds2)

    mlflow.log_metric("rmse", rmse2)
    mlflow.log_metric("mae",  mae2)
    mlflow.log_metric("r2",   r2_2)
    mlflow.log_metric("explained_variance", var2)

    mlflow.sklearn.log_model(gbt, "gbt_model")

    # Artifact 1
    importance2 = pd.DataFrame({
        "Feature": feature_cols,
        "Importance": gbt.feature_importances_
    }).sort_values("Importance", ascending=False)
    tmp3 = tempfile.NamedTemporaryFile(suffix=".csv", delete=False)
    importance2.to_csv(tmp3.name, index=False)
    mlflow.log_artifact(tmp3.name, "feature_importance.csv")
    os.unlink(tmp3.name)

    # Artifact 2: Residual plot
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(preds2, y_test - preds2, alpha=0.3)
    ax.axhline(0, color="red", linestyle="--")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Residual")
    ax.set_title("Residual Plot — Gradient Boosting")
    tmp4 = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
    fig.savefig(tmp4.name); plt.close(fig)
    mlflow.log_artifact(tmp4.name, "residual_plot.png")
    os.unlink(tmp4.name)

    print(f"✅ Model 2 - Gradient Boosting")
    print(f"   RMSE: {rmse2:.4f} | MAE: {mae2:.4f} | R²: {r2_2:.4f} | Var: {var2:.4f}")


In [0]:
from pyspark.sql.functions import lit, current_timestamp

test_meta = df.loc[X_test.index, ["raceId", "driverId", "finishing_position"]].copy()

# Model 1
pred_df1 = test_meta.copy()
pred_df1["predicted_position"] = preds1
pred_df1["model_name"] = "random_forest"

spark.createDataFrame(pred_df1) \
    .withColumn("raceId",             F.col("raceId").cast("int")) \
    .withColumn("driverId",           F.col("driverId").cast("int")) \
    .withColumn("actual_position",    F.col("finishing_position").cast("int")) \
    .withColumn("predicted_position", F.col("predicted_position").cast("double")) \
    .withColumn("created_at",         current_timestamp()) \
    .select("raceId","driverId","actual_position","predicted_position","model_name","created_at") \
    .write.mode("overwrite") \
    .saveAsTable("`gr5069`.`sl5919`.`model1_predictions`")

# Model 2
pred_df2 = test_meta.copy()
pred_df2["predicted_position"] = preds2
pred_df2["model_name"] = "gbt"

spark.createDataFrame(pred_df2) \
    .withColumn("raceId",             F.col("raceId").cast("int")) \
    .withColumn("driverId",           F.col("driverId").cast("int")) \
    .withColumn("actual_position",    F.col("finishing_position").cast("int")) \
    .withColumn("predicted_position", F.col("predicted_position").cast("double")) \
    .withColumn("created_at",         current_timestamp()) \
    .select("raceId","driverId","actual_position","predicted_position","model_name","created_at") \
    .write.mode("overwrite") \
    .saveAsTable("`gr5069`.`sl5919`.`model2_predictions`")

print("✅ Predictions written to:")
print("  - gr5069.sl5919.model1_predictions")
print("  - gr5069.sl5919.model2_predictions")

spark.sql("SELECT model_name, COUNT(*) as n FROM `gr5069`.`sl5919`.`model1_predictions` GROUP BY model_name").show()
spark.sql("SELECT model_name, COUNT(*) as n FROM `gr5069`.`sl5919`.`model2_predictions` GROUP BY model_name").show()